# Lab 05 — Augmented Prompts: The RAG Core Loop

**Knowledge base:**
- `knowledge_base/01_rag_fundamentals/01_what_is_rag.md`
- `knowledge_base/01_rag_fundamentals/03_rag_architecture.md`

**Concepts:** Retrieve → Augment → Generate · context injection · hallucination vs grounding

This is the central lab of module 01. You will implement the complete RAG pipeline
using a hardcoded knowledge base (no vector database yet).

---

In [ ]:
import os, sys
sys.path.insert(0, ".")
from dotenv import load_dotenv
from utils import generate_with_single_input, generate_with_multiple_input

load_dotenv(override=True)
print(f"✅ Backend: {os.environ.get('LLM_BACKEND', 'ollama')}")

---
## 1 — The three-step RAG loop

Every RAG system, no matter how sophisticated, follows this loop:

```
User query
    │
    ▼
[RETRIEVE]  → search the knowledge base → return top-K documents
    │
    ▼
[AUGMENT]   → build a prompt = query + retrieved documents
    │
    ▼
[GENERATE]  → send augmented prompt to LLM → get grounded answer
```

In this lab the retriever is a simple Python function. In module 02 you will replace it with BM25, semantic search, and hybrid search.

---
## 2 — The knowledge base

A knowledge base is a list of documents. In production it's a vector database.
Here it's a Python list so we can focus on the loop, not the infrastructure.

In [ ]:
# Knowledge base — 10 facts about Egypt and the Suez Canal
# These are the ONLY facts the LLM is allowed to use
KNOWLEDGE_BASE = [
    "Egypt is a country in northeastern Africa with a population of approximately 105 million people.",
    "Cairo is the capital and largest city of Egypt, located on the banks of the Nile River.",
    "The Suez Canal was inaugurated on November 17, 1869, after 10 years of construction.",
    "The Suez Canal is 193 kilometres long and connects the Red Sea to the Mediterranean Sea.",
    "Ferdinand de Lesseps was the French diplomat who led the construction of the Suez Canal.",
    "Egypt nationalized the Suez Canal in 1956 under President Gamal Abdel Nasser.",
    "The Nile River is approximately 6,650 km long, making it one of the longest rivers in the world.",
    "Egypt's official language is Arabic. The country uses the Egyptian pound as its currency.",
    "The Great Pyramid of Giza is the oldest and largest of the Seven Wonders of the Ancient World.",
    "Suez Canal University (SCU) was established in Ismailia, Egypt, in 1976.",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
for i, doc in enumerate(KNOWLEDGE_BASE):
    print(f"  [{i}] {doc[:80]}...")

---
## 3 — Step 1: Retrieve

The retriever's job is to find the documents most relevant to the query.
This naive retriever scores by counting shared words. In module 02 you'll replace it with proper algorithms.

In [ ]:
def simple_retriever(query: str, knowledge_base: list, top_k: int = 3) -> list:
    """
    Naive keyword-overlap retriever.
    Scores each document by how many words from the query it contains.
    Returns the top_k highest-scoring documents.

    Args:
        query:          The user's question.
        knowledge_base: List of document strings.
        top_k:          Number of documents to return.

    Returns:
        List of (score, document) tuples, sorted descending by score.
    """
    query_words = set(query.lower().split())
    scored = []
    for doc in knowledge_base:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(reverse=True)
    return scored[:top_k]


# Test the retriever
results = simple_retriever("When was the Suez Canal opened?", KNOWLEDGE_BASE, top_k=3)
print("Top-3 retrieved documents:\n")
for score, doc in results:
    print(f"  [overlap={score}] {doc}")

---
## 4 — Step 2: Augment

Augmentation = injecting the retrieved documents into the prompt.
The user's query goes in. The retrieved documents go in. One prompt comes out.

In [ ]:
def build_augmented_prompt(query: str, retrieved_docs: list) -> str:
    """
    Build the augmented prompt from a query and retrieved (score, doc) pairs.

    Args:
        query:          The user's original question.
        retrieved_docs: List of (score, doc) tuples from the retriever.

    Returns:
        The full augmented prompt string.
    """
    # Format the documents into a numbered context block
    context_lines = [f"Document {i+1}: {doc}" for i, (score, doc) in enumerate(retrieved_docs)]
    context = "\n".join(context_lines)

    return f"""You are a helpful assistant. Answer the question using ONLY the documents provided below.
If the answer is not in the documents, say: "I don't know based on the provided documents."

Retrieved Documents:
{context}

Question: {query}

Answer:"""


query = "When was the Suez Canal opened?"
retrieved = simple_retriever(query, KNOWLEDGE_BASE, top_k=3)
prompt = build_augmented_prompt(query, retrieved)
print(prompt)

---
## 5 — Step 3: Generate

Send the augmented prompt to the LLM. The model reads the retrieved documents and answers from them.

In [ ]:
# The full RAG loop in 3 lines
query     = "When was the Suez Canal opened?"
retrieved = simple_retriever(query, KNOWLEDGE_BASE, top_k=3)   # RETRIEVE
prompt    = build_augmented_prompt(query, retrieved)            # AUGMENT
response  = generate_with_single_input(prompt=prompt, temperature=0.0)  # GENERATE

print(f"Question: {query}")
print(f"Answer:   {response['content']}")

In [ ]:
# Wrap it into a single function — this IS a minimal RAG system
def rag(query: str, knowledge_base: list, top_k: int = 3) -> dict:
    """
    Full RAG pipeline: retrieve → augment → generate.

    Args:
        query:          The user's question.
        knowledge_base: List of document strings.
        top_k:          Number of documents to retrieve.

    Returns:
        Dict with 'answer', 'retrieved_docs', and 'prompt' keys.
    """
    retrieved = simple_retriever(query, knowledge_base, top_k=top_k)
    prompt    = build_augmented_prompt(query, retrieved)
    response  = generate_with_single_input(prompt=prompt, temperature=0.0)

    return {
        "answer":         response["content"],
        "retrieved_docs": retrieved,
        "prompt":         prompt,
    }


# Test it on several questions
questions = [
    "What is the capital of Egypt?",
    "Who built the Suez Canal?",
    "What happened to the Suez Canal in 1956?",
    "Where is Suez Canal University located?",
]

for q in questions:
    result = rag(q, KNOWLEDGE_BASE)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print()

---
## 6 — Hallucination vs grounding

This is the most important demonstration in module 01.
Ask the LLM the same question **without** RAG and **with** RAG.
Watch what happens when there is no context.

In [ ]:
question = "What year was Suez Canal University established and in which city?"

# WITHOUT RAG — the LLM answers from its training data (may hallucinate)
bare_response = generate_with_single_input(
    prompt=question,
    temperature=0.0
)
print("WITHOUT RAG (bare LLM):")
print(bare_response['content'])
print()

# WITH RAG — the LLM reads the retrieved documents and answers from them
rag_result = rag(question, KNOWLEDGE_BASE)
print("WITH RAG (grounded by retrieved documents):")
print(rag_result['answer'])

In [ ]:
# Ask about something NOT in the knowledge base — observe graceful refusal
question_outside = "What is the population of Sudan?"
result = rag(question_outside, KNOWLEDGE_BASE)
print(f"Q: {question_outside}")
print(f"A: {result['answer']}")
print()
print("Retrieved documents for this query:")
for score, doc in result['retrieved_docs']:
    print(f"  [overlap={score}] {doc[:80]}...")

---
## 7 — Exercise: extend the knowledge base and test

Add 3 new documents to `KNOWLEDGE_BASE` on a topic of your choice,
then ask the RAG system 2 questions that can only be answered from those new documents.

In [ ]:
# Add your 3 documents here
MY_KB = KNOWLEDGE_BASE + [
    # YOUR DOCUMENT 1
    # YOUR DOCUMENT 2
    # YOUR DOCUMENT 3
]

# Ask 2 questions that require your new documents
my_questions = [
    # YOUR QUESTION 1
    # YOUR QUESTION 2
]

for q in my_questions:
    result = rag(q, MY_KB)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print()

---
**Lab 05 complete.**

You have implemented a working RAG system: retrieve → augment → generate.
You saw the difference between a hallucinated answer and a grounded one.

The retriever in this lab is naive — it only counts word overlap.
**Module 02** replaces it with TF-IDF, BM25, semantic search, and hybrid search.

**Next:** `lab06_manual_retrieval.ipynb` — understand why naive retrieval fails
and what module 02 will fix.